# 🧠 GODS 4.0 Competition — PowerPointPoys (22nd Place Globally)

## 🏆 Overview

This repository contains our submission for the **GODS 4.0 international AI competition**, which focused on detecting early signs of **mental health issues** from user-generated text. Our solution ranked **22nd worldwide** with a public leaderboard score of **0.778**, making it the **best-performing submission by any first-year team** in the entire competition.

We used a transformer-based approach leveraging **DistilBERT** for sequence classification, combined with careful preprocessing, tokenization, and fine-tuning.

---

## 🧠 Competition Background

> Mental health awareness is more crucial than ever, with millions of individuals sharing their struggles online. The ability to detect early signs of mental health issues from text-based content can be an essential step in providing timely support and intervention.

Participants were challenged to build machine learning models that classify short text snippets into different mental health-related categories such as:

- **Depression**
- **Anxiety**
- **Relationship-related issues**
- **Work and Academic pressure**, and more.

---

## 🧰 Our Approach

We designed a complete NLP pipeline using **Hugging Face Transformers**, with `distilbert-base-uncased` as the backbone model. Below are the key components of our solution:

### 🔄 Data Preprocessing
- Filled missing values in `title` and `content`
- Combined both fields into a unified `text` field
- Encoded target labels into numerical values

### 📁 Dataset Preparation
- Used `train_test_split` for 80/20 data partitioning
- Converted data into `Hugging Face DatasetDict` format for compatibility

### 🔠 Tokenization
- Used the **DistilBERT tokenizer** with:
  - Padding
  - Truncation (`max_length=512`)
  - Batch tokenization for efficiency

### 🤖 Model Architecture
- Fine-tuned a `DistilBERT` model (`AutoModelForSequenceClassification`)
- Output layer adapted to match number of unique labels

### 🧪 Evaluation
- Tracked accuracy, F1-score, and confusion matrix
- Conducted predictions on the test set for final submission

---

## 📊 Results

- 🥇 **Final Score**: `0.778`
- 🥈 **Public Leaderboard Rank**: 22 / ~1,000+
- 🥉 **Top Rank Among First-Year Students** 🎉

Our result demonstrates the potential of transformer-based models in real-world mental health detection tasks, especially when combined with good preprocessing and training strategies.

---

## 👥 Team

**PowerPointPoys** – A team of first-year CS engineering students with a passion for impactful AI.

---



In [ ]:
pip install transformers datasets torch scikit-learn tqdm


In [ ]:
# Install Required Libraries
Install the necessary libraries for the project, including `transformers`, `datasets`, `torch`, `scikit-learn`, and `tqdm`.

In [ ]:
!pip install --upgrade transformers
!pip install datasets


In [ ]:
# Load and Preprocess Dataset
Load the dataset from Google Drive.

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# Load your dataset
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/My Drive/gods/gods/train.csv')



In [ ]:
# Preprocess Dataset
In this step, we preprocess the dataset to prepare it for training. The preprocessing involves several tasks:

1. **Handle Missing Values**: Fill any missing values in the `title` and `content` columns with empty strings to ensure data consistency.
2. **Combine Text Fields**: Merge the `title` and `content` columns into a single `text` column to create a unified input for the model.
3. **Encode Target Labels**: Map the target labels to numerical values using a label encoding approach, which is essential for classification tasks.
4. **Split the Dataset**: Divide the dataset into training and validation sets to evaluate the model's performance during training.
5. **Convert to Hugging Face Dataset Format**: Transform the processed data into the Hugging Face `Dataset` format, which is compatible with the `transformers` library and simplifies further processing steps.

In [ ]:
# Fill NaN values
df["title"] = df["title"].fillna("")
df["content"] = df["content"].fillna("")

# Combine title and content
df["text"] = df["title"] + " " + df["content"]

# Encode target labels
label_mapping = {label: i for i, label in enumerate(df["target"].unique())}
df["label"] = df["target"].map(label_mapping)

# Split dataset
train_texts, val_texts, train_labels, val_labels = train_test_split(df["text"], df["label"], test_size=0.2, random_state=42)

# Convert to Hugging Face Dataset format
train_data = Dataset.from_dict({"text": train_texts.tolist(), "label": train_labels.tolist()})
val_data = Dataset.from_dict({"text": val_texts.tolist(), "label": val_labels.tolist()})

# Define dataset dictionary
dataset = DatasetDict({"train": train_data, "validation": val_data})

In [ ]:
# Tokenize Dataset
In this step, we tokenize the dataset to prepare it for input into the model. The process involves:

1. **Load Pretrained Tokenizer**: Use the `distilbert-base-uncased` tokenizer from the Hugging Face `transformers` library.
2. **Define Tokenization Function**: Create a function to tokenize the `text` field of the dataset, applying padding and truncation to ensure uniform input size.
3. **Apply Tokenization**: Use the `map` method to apply the tokenization function to the entire dataset in a batched manner, resulting in a tokenized dataset ready for training and evaluation.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Apply tokenization
tokenized_datasets = dataset.map(tokenize_function, batched=True)


In [ ]:
# Define Model
In this step, we define the model for sequence classification:

1. **Determine Number of Labels**: Calculate the number of unique target labels in the dataset to set the output size of the model.
2. **Load Pretrained Model**: Use the `distilbert-base-uncased` model from the Hugging Face `transformers` library, specifying the number of labels for the classification task.

In [ ]:
num_labels = len(df["target"].unique())
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=num_labels)


In [ ]:
# Define Training Arguments
In this step, we configure the training arguments for the model. These arguments include:

1. **Output Directory**: Specify the directory where the model checkpoints and results will be saved.
2. **Evaluation and Save Strategy**: Set the evaluation and checkpoint saving to occur at the end of each epoch.
3. **Batch Size**: Define the batch size for both training and evaluation to optimize GPU usage.
4. **Number of Epochs**: Set the number of training epochs.
5. **Weight Decay**: Apply weight decay for regularization.
6. **Mixed Precision Training**: Enable FP16 for faster training on GPUs.
7. **Logging**: Specify the logging directory and disable reporting to external services.

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    per_device_train_batch_size=16,  # Increase batch size (8 → 16) to use more GPU power
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,  # Enable mixed precision (faster on GPU)
    report_to="none",
)


In [ ]:
# Train the Model
In this step, we train the model using the Hugging Face `Trainer` class. The process involves:

1. **Initialize Trainer**: Create an instance of the `Trainer` class, specifying the model, training arguments, and datasets for training and evaluation.
2. **Train the Model**: Call the `train` method to start the training process, which optimizes the model's parameters based on the training data.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"]
)

trainer.train()


In [ ]:
# Evaluate the Model
In this step, we evaluate the trained model on the validation dataset. The evaluation process involves:

1. **Call the `evaluate` Method**: Use the `evaluate` method of the `Trainer` class to compute evaluation metrics such as loss and accuracy on the validation set.
2. **Analyze Results**: Review the evaluation results to assess the model's performance and identify areas for improvement.

In [ ]:
trainer.evaluate()


In [ ]:
# Get Predictions and Compute Accuracy
In this step, we evaluate the model's predictions on the validation set and compute the accuracy:

1. **Generate Predictions**: Use the `predict` method of the `Trainer` class to obtain predictions on the validation dataset.
2. **Determine Predicted Labels**: Apply the `argmax` function to extract the predicted class labels from the model's output logits.
3. **Compute Accuracy**: Use the `accuracy_score` function from `sklearn.metrics` to calculate the accuracy of the predictions compared to the true labels.
4. **Display Results**: Print the validation accuracy to assess the model's performance.

In [ ]:
# Get predictions on validation set
from sklearn.metrics import accuracy_score
preds_output = trainer.predict(tokenized_datasets["validation"])
preds = torch.argmax(torch.tensor(preds_output.predictions), dim=-1)

# Compute accuracy
accuracy = accuracy_score(val_labels, preds.numpy())
print(f"Validation Accuracy: {accuracy:.4f}")


In [ ]:
# Predict on Test Data
In this step, we process the test dataset and generate predictions:

1. **Load Test Dataset**: Read the `test.csv` file containing the test data.
2. **Handle Missing Values**: Fill any missing values in the `title` and `content` columns with empty strings.
3. **Combine Text Fields**: Merge the `title` and `content` columns into a single `text` column for input to the model.
4. **Convert to Dataset Format**: Transform the test data into the Hugging Face `Dataset` format for compatibility with the tokenizer and model.
5. **Tokenize Test Data**: Apply the tokenization function to the test dataset.
6. **Generate Predictions**: Use the trained model to predict the labels for the test dataset.
7. **Map Predictions to Original Labels**: Convert the predicted numerical labels back to their original string representations.

In [ ]:
# Load test.csv
test_df = pd.read_csv('/content/drive/My Drive/gods/gods/test.csv')

# Fill NaN values and combine title + content
test_df["title"] = test_df["title"].fillna("")
test_df["content"] = test_df["content"].fillna("")
test_df["text"] = test_df["title"] + " " + test_df["content"]

# Convert test data into Dataset format
test_dataset = Dataset.from_dict({"text": test_df["text"].tolist()})
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Get predictions
preds_output = trainer.predict(tokenized_test)
predictions = torch.argmax(torch.tensor(preds_output.predictions), dim=-1)

# Convert back to original label names
predicted_labels = [list(label_mapping.keys())[pred] for pred in predictions.numpy()]


In [ ]:
# Create Submission File
In this step, we prepare the submission file for the test dataset:

1. **Create Submission DataFrame**: Combine the `id` column from the test dataset with the predicted labels to form the submission DataFrame.
2. **Saving to CSV**: Export the submission DataFrame to a CSV file named `submission.csv`.
3. **Confirm Save**: Print a confirmation message indicating that the submission file has been saved successfully.

In [ ]:
# Create submission DataFrame
submission = pd.DataFrame({"id": test_df["id"], "target": predicted_labels})

# Save to CSV
submission.to_csv("submission.csv", index=False)

print("Submission file saved as submission.csv!")
